# Adaptive Subject-Level Differential Privacy — Colab Runner

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook clones the repo, installs dependencies, and runs the full experiment.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/shaktsin/ml-research.git
%cd ml-research/adaptive-diff-privacy

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Verify GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Run the experiment

Runs all 3 configs: Baseline, Subject-DP (uniform), Adaptive Subject-DP.

In [ ]:
!python run_experiment.py

## 5. (Optional) Tweak parameters and re-run

Edit the knobs below and re-run a single config without re-running everything.

In [ ]:
import sys
sys.path.insert(0, '.')

from data import get_dataloaders
from model import get_model
from trainer import train_baseline, train_subject_dp, evaluate
from mia import run_mia

# --- Knobs ---
EPOCHS     = 3
CLIP_NORM  = 1.0
BASE_SIGMA = 1.0
N_SUBJECTS = 500
MAX_TRAIN  = 2000   # set None for full 120k dataset (much slower)
BATCH_SIZE = 16

train_loader, test_loader = get_dataloaders(
    max_train=MAX_TRAIN, batch_size=BATCH_SIZE, n_subjects=N_SUBJECTS
)

print('Data loaded.')
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# Run only Adaptive Subject-DP
model = get_model(device=device)
train_subject_dp(model, train_loader, epochs=EPOCHS,
                 clip_norm=CLIP_NORM, base_sigma=BASE_SIGMA, adaptive=True)
acc = evaluate(model, test_loader)
auc = run_mia(model, train_loader, test_loader)
print(f'Accuracy: {acc:.4f} | MIA AUC: {auc:.4f}')